# Cognilend: Hybrid Loan-Approval Engine (rules + ML)

The first notebook (`loan_approval_prediction.ipynb`) followed the video: clean, label-encode, scale, try four classifiers and pick the most accurate one. That reached **~84% accuracy**, and it treated a loan decision as a plain classification problem.

A real bank decision has a structure, and this notebook builds a system that mirrors it:

| Layer | What it does | Why |
|---|---|---|
| **Cleaning + imputation** | Fixes spellings, units and sign errors. Imputation is fitted on training data only, and every fix is flagged | No leakage, and uncertain data can be traced |
| **Policy-aware features** | FOIR against its income-tiered limit, LTV against the RBI cap, age at maturity against the segment limit, plus the *headroom* to each limit | Underwriting decides on ratios and limits, not raw rupees |
| **Rule engine R1–R7** | Hard policy, independent of the model | A model must never approve what policy forbids |
| **Gradient-boosted model** | Monotone-constrained XGBoost, tuned with Optuna | Accuracy, plus the guarantee that a better CIBIL score can never lower the score |
| **Decision layer** | APPROVE / REJECT / MANUAL_REVIEW, each with written reasons | Every scenario is handled, and RBI's Fair Practices Code asks for reasons for rejection |
| **Fairness audit** | DIR, approval-rate CIs and TPR gap by gender and area | Gender and marital status are never model inputs, and the audit checks the outcome |

The code is in `loan_engine.py` (engine) and `train.py` (training, audit and saving). `test_scenarios.py` holds 51 scenario tests.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score, log_loss, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
from sklearn.calibration import calibration_curve

import loan_engine as le
import train as T

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 200)

## 1. Load and clean (leak-free)

In [ ]:
raw = pd.read_csv('loan_applications_raw.csv')
df = le.prepare_training_data(raw)      # strip, dedupe, drop unlabelled, spellings, dtypes, validity fixes
print('raw', raw.shape, '-> cleaned', df.shape, '| approval rate', round(df.target.mean(), 3))
print('rows with at least one data-quality fix:', (df.dq_fixes > 0).sum())
df[le.NUMERIC_COLS].isna().sum().to_frame('missing (imputed later, inside the training fold)').T

In [ ]:
# One fixed hold-out test set, never touched during model selection or tuning
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
imp = le.Imputer().fit(train_df)                    # medians/modes learned on train only
F_train = le.build_features(imp.transform(train_df))
F_test = le.build_features(imp.transform(test_df))
print(F_train.shape, F_test.shape)

## 2. How decisions were actually made: the policy rules

The historical label was produced in two stages, hard policy rules and then an underwriter's judgement. The rule engine reproduces the report's violation counts exactly.

In [ ]:
F_all = le.build_features(le.Imputer().fit(df).transform(df))
viol = le.evaluate_rules(F_all)
summary = pd.DataFrame({'violations': viol.sum(),
                        'share': viol.mean().round(3),
                        'approved anyway': [F_all.loc[viol[r] == 1, 'target'].mean().round(3) for r in le.RULES]})
summary.index = [f'{r}: {le.RULE_TEXT[r].split(" ")[0]}...' for r in le.RULES]
any_v = viol.sum(axis=1) > 0
print(f'any violation: {any_v.sum()} ({any_v.mean():.1%}), approved anyway: {F_all.target[any_v].mean():.1%}')
print(f'pass all rules: approval rate {F_all.target[~any_v].mean():.1%}')
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
F_all['n_rules_violated'].clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'}).pipe(
    lambda s: F_all.groupby(s)['target'].mean()).plot.bar(ax=axes[0], color='#4C72B0')
axes[0].set(title='Approval rate by number of rules violated', xlabel='rules violated', ylabel='approval rate')
ok = F_all[~any_v]
pd.cut(ok['cibil_filled'], [649, 700, 750, 800, 900]).pipe(lambda s: ok.groupby(s, observed=True)['target'].mean()).plot.bar(ax=axes[1], color='#55A868')
axes[1].set(title='Policy-compliant: approval by CIBIL', xlabel='CIBIL', ylabel='')
pd.cut(ok['foir_headroom'], [-0.01, 0.1, 0.2, 0.3, 0.5, 1]).pipe(lambda s: ok.groupby(s, observed=True)['target'].mean()).plot.bar(ax=axes[2], color='#C44E52')
axes[2].set(title='Policy-compliant: approval by FOIR headroom', xlabel='limit - FOIR', ylabel='')
plt.tight_layout(); plt.show()

**Reading:** a single violation already cuts approval to about 4%. Among compliant applicants, approval climbs steadily with CIBIL and with the room left under the FOIR limit. These two effects are what the features below encode.

## 3. Feature engineering: encode the policy, not just the raw columns

All ratios are built from real, unscaled and uncapped values (report §3.8 ordering pitfall).

| Feature | Meaning |
|---|---|
| `foir`, `foir_headroom` | (existing EMI + new EMI) / income, and its distance to the income-tiered limit (50/55/65%) |
| `ltv_headroom`, `collateral_coverage` | Distance to the RBI LTV slab cap for home loans, and collateral ÷ loan |
| `loan_to_income`, `total_income` | Loan size in years of income, and clubbed income |
| `cibil_filled`, `is_new_to_credit` | Score, with new-to-credit applicants as their own branch instead of a fake median |
| `age_at_maturity`, `maturity_headroom` | Age at the last EMI, and years left before the segment's limit (60 or 65) |
| `Work_Experience`, `experience_headroom` | Vintage, and the margin over the segment minimum (1 or 3 years) |
| `n_rules_violated` | How many hard rules are broken |
| segment / product dummies, `Dependents`, `Age`, `Loan_Tenure` | Remaining credit factors |

**Excluded on purpose:** `Gender` and `Marital_Status` (protected) and `Property_Area` (no measurable gain, and a possible proxy).

## 4. Model selection with leak-free cross-validation

Every fold re-fits the imputer on its own training part. The three feature sets compared are the video's raw columns, the report's 14 features, and the engine's policy-aware features.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

skf = StratifiedKFold(5, shuffle=True, random_state=0)
folds = []
for a, b in skf.split(train_df, train_df.target):
    A, B = train_df.iloc[a], train_df.iloc[b]
    fi = le.Imputer().fit(A)
    FA, FB = le.build_features(fi.transform(A)), le.build_features(fi.transform(B))
    for X in (FA, FB):
        X['cibil_median_filled'] = X['cibil_filled'].fillna(fi.cibil_median_)
        X['loan_amount_log'] = np.log(X['Loan_Amount']); X['total_income_log'] = np.log(X['total_income'])
    folds.append((FA, FB))

def cv(features, make_model):
    acc, auc, ll = [], [], []
    for A, B in folds:
        m = make_model()
        # Logistic Regression / Random Forest cannot take NaN (new-to-credit CIBIL): give them -1 instead
        prep = (lambda X: X.fillna(-1)) if not hasattr(m, 'get_booster') and not isinstance(m, (HistGradientBoostingClassifier, LGBMClassifier)) else (lambda X: X)
        m.fit(prep(A[features]), A.target)
        p = m.predict_proba(prep(B[features]))[:, 1]
        acc.append(accuracy_score(B.target, p >= .5)); auc.append(roc_auc_score(B.target, p)); ll.append(log_loss(B.target, p))
    return np.mean(acc), np.mean(auc), np.mean(ll)

video_feats = ['Age', 'Dependents', 'emp_sep', 'emp_senp', 'Work_Experience', 'Existing_EMI', 'cibil_median_filled',
               'loan_type_home', 'loan_type_vehicle', 'loan_type_business', 'Loan_Tenure', 'Collateral_Value',
               'area_urban', 'area_rural', 'loan_amount_log', 'total_income_log']
report_feats = ['Work_Experience', 'total_income', 'cibil_median_filled', 'is_new_to_credit', 'Loan_Tenure', 'foir',
                'collateral_coverage', 'loan_to_income', 'age_at_maturity', 'emp_senp', 'emp_sep',
                'loan_type_business', 'loan_type_home', 'loan_type_vehicle']
engine_feats = le.MODEL_FEATURES
models = {
    'Logistic Regression': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'Random Forest': lambda: RandomForestClassifier(500, min_samples_leaf=3, n_jobs=-1, random_state=0),
    'HistGradientBoosting': lambda: HistGradientBoostingClassifier(random_state=0),
    'LightGBM': lambda: LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, verbose=-1, random_state=0),
    'XGBoost (tuned, monotone)': lambda: T.make_model(),
}
rows = []
for fs_name, fs in [('video raw columns', video_feats), ('report 14', report_feats), ('engine (policy-aware)', engine_feats)]:
    for m_name, mk in models.items():
        if m_name.startswith('XGBoost') and fs is not engine_feats:
            continue
        a, u, l = cv(fs, mk)
        rows.append({'features': fs_name, 'model': m_name, 'CV accuracy': a, 'CV ROC-AUC': u, 'CV log-loss': l})
cmp = pd.DataFrame(rows).round(4)
cmp.pivot(index='model', columns='features', values='CV accuracy')

In [ ]:
cmp.sort_values('CV log-loss').head(8)

## 5. Feature selection: three complementary checks

1. **Mutual information**: relevance of each feature on its own.
2. **Permutation importance** on a validation fold: what the model actually relies on.
3. **Backward elimination** on cross-validated log-loss: drop a feature only if the model gets measurably better without it. In the tuning run, no removal improved log-loss by more than 0.0005, so all features were kept.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance

X_tr, y_tr = F_train[engine_feats], F_train['target']
mi = pd.Series(mutual_info_classif(X_tr.fillna(-1), y_tr, random_state=0), index=engine_feats)
A, B = folds[0]
m = T.make_model().fit(A[engine_feats], A.target)
pi = permutation_importance(m, B[engine_feats], B.target, scoring='neg_log_loss', n_repeats=5, random_state=0)
imp_tbl = pd.DataFrame({'mutual_info': mi, 'permutation_importance': pi.importances_mean}).sort_values('permutation_importance', ascending=False)
imp_tbl.round(4)

In [ ]:
# Correlation check: tree models tolerate correlated features, but they spread importance between them
corr = F_train[engine_feats].corr(method='spearman').abs()
pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().sort_values(ascending=False)
pairs.head(8).round(3).to_frame('|spearman rho|')

## 6. Final model on the untouched test set

In [ ]:
engine = le.LoanDecisionEngine(T.make_model()).fit(train_df, train_df['target'])
p_test = engine.predict_proba(test_df)
y_test = test_df['target'].to_numpy()
print(f'Test accuracy {accuracy_score(y_test, p_test >= .5):.4f} | ROC-AUC {roc_auc_score(y_test, p_test):.4f} | '
      f'F1 {f1_score(y_test, p_test >= .5):.4f} | log-loss {log_loss(y_test, p_test):.4f}')
print(classification_report(y_test, p_test >= .5, target_names=['Rejected', 'Approved']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, p_test >= .5), display_labels=['Rejected', 'Approved']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion matrix (threshold 0.5)')
RocCurveDisplay.from_predictions(y_test, p_test, ax=axes[1], name='XGBoost (engine)')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1); axes[1].set_title('ROC curve')
frac, mean_p = calibration_curve(y_test, p_test, n_bins=10)
axes[2].plot(mean_p, frac, 'o-'); axes[2].plot([0, 1], [0, 1], 'k--', lw=1)
axes[2].set(title='Calibration: predicted vs actual approval', xlabel='predicted P(approve)', ylabel='observed rate')
plt.tight_layout(); plt.show()

**How good can it get?** The labels were *sampled* from an underwriter probability (report §1.1), so some outcomes are coin flips that no model can predict. If the model's probabilities are well calibrated, as the curve above shows, then `mean(max(p, 1-p))` estimates the best accuracy reachable with these inputs. Scoring close to that number means there is no real accuracy left to gain. More gain could only come from fitting noise, or from using gender, which is excluded.

In [ ]:
expected = np.mean(np.maximum(p_test, 1 - p_test))
print(f'Expected accuracy if calibrated: {expected:.4f}  |  achieved: {accuracy_score(y_test, p_test >= .5):.4f}')

## 7. Explainability: what drives the score (SHAP)

In [ ]:
import shap
sv = engine.contributions(engine.featurize(test_df))
shap.summary_plot(sv.to_numpy(), engine.featurize(test_df)[engine.features], show=False, max_display=15)
plt.title('SHAP: impact on log-odds of approval'); plt.tight_layout(); plt.show()

## 8. The hybrid decision: every scenario has a defined outcome

| Order | Situation | Outcome |
|---|---|---|
| 1 | Any of R1–R7 violated | **REJECT** with the rule text. If the model score is high, a note says a deviation would need sanctioning-authority approval |
| 2 | New to credit; CIBIL, EMI, income, age, amount, tenure or product missing or invalid; collateral not on record for a secured loan; exposure ≥ ₹2 Cr | **MANUAL_REVIEW** with the model's recommendation attached |
| 3 | P(approve) ≥ 0.65 | **APPROVE** with the top supporting factors |
| 4 | P(approve) < 0.35 | **REJECT** with the top adverse factors (written reasons) |
| 5 | Otherwise | **MANUAL_REVIEW**: borderline, with concerns and strengths listed |

In [ ]:
decisions = engine.decide(test_df)
auto = decisions['decision'] != 'MANUAL_REVIEW'
auto_acc = accuracy_score(y_test[auto.to_numpy()], (decisions.loc[auto, 'decision'] == 'APPROVE').astype(int))
print(f'Auto-decided: {auto.mean():.1%} of applications, accuracy on those {auto_acc:.4f}')
print(f'Manual review: {(~auto).mean():.1%}')
pd.crosstab([decisions['decision'], decisions['decided_by']], test_df['Loan_Status'].to_numpy(), margins=True)

In [ ]:
# Trade-off: widening the review band raises auto-decision accuracy but sends more files to underwriters
F_t = engine.featurize(test_df)
policy_rej = le.evaluate_rules(F_t).sum(axis=1).to_numpy() > 0
trig = np.array([bool(le.review_triggers(r)) for _, r in F_t.iterrows()])
rows = []
for band in [0, 0.05, 0.10, 0.15, 0.20, 0.25]:
    lo, hi = 0.5 - band, 0.5 + band
    pred = np.where(policy_rej, 0, (p_test >= 0.5).astype(int))
    review = ~policy_rej & (trig | ((p_test >= lo) & (p_test < hi)))
    rows.append({'review band': f'{lo:.2f}-{hi:.2f}', 'manual review %': review.mean() * 100,
                 'auto accuracy': accuracy_score(y_test[~review], pred[~review])})
pd.DataFrame(rows).round(3)

### 8.1 Scenario walkthrough: one applicant per path

In [ ]:
base = dict(Age=35, Gender='Male', Marital_Status='Married', Dependents='1', Employment_Type='Salaried',
            Work_Experience=10, Monthly_Income=150000, Coapplicant_Income=None, Existing_EMI=0, CIBIL_Score=800,
            Loan_Type='Home Loan', Loan_Amount=3000000, Loan_Tenure=240, Collateral_Value=5000000, Property_Area='Urban')
scenarios = {
    'A. Strong salaried home-loan applicant':          {},
    'B. Same applicant, female (fairness check)':      dict(Gender='Female'),
    'C. Under-age (R1)':                               dict(Age=20, Work_Experience=1),
    'D. Loan runs past retirement (R2)':               dict(Age=48, Loan_Tenure=240),
    'E. CIBIL 640 (R3)':                               dict(CIBIL_Score=640),
    'F. Over-leveraged: FOIR above limit (R4)':        dict(Monthly_Income=45000, Existing_EMI=12000),
    'G. Home loan above RBI LTV cap (R5)':             dict(Collateral_Value=3200000),
    'H. Income below Rs 15k (R6)':                     dict(Monthly_Income=12000, Loan_Type='Personal Loan', Loan_Amount=50000, Loan_Tenure=24, Collateral_Value=None),
    'I. New business, 2-yr vintage (R7)':              dict(Employment_Type='SENP', Work_Experience=2),
    'J. Several violations at once':                   dict(Age=20, CIBIL_Score=600, Work_Experience=0, Monthly_Income=12000),
    'K. New to credit (no bureau history)':            dict(CIBIL_Score=-1, Age=24, Work_Experience=2),
    'L. CIBIL missing from file':                      dict(CIBIL_Score=None),
    'M. Home loan, collateral value not on record':    dict(Collateral_Value=None),
    'N. Rs 2.5 Cr exposure':                           dict(Monthly_Income=1500000, Loan_Amount=25000000, Collateral_Value=40000000),
    'O. Compliant but weak (low CIBIL, thin margins)': dict(CIBIL_Score=660, Monthly_Income=32000, Existing_EMI=6000, Work_Experience=1,
                                                            Loan_Type='Business Loan', Loan_Amount=250000, Loan_Tenure=36, Collateral_Value=None, Dependents='3+'),
    'P. Borderline personal loan':                     dict(CIBIL_Score=700, Monthly_Income=45000, Existing_EMI=4000, Work_Experience=3,
                                                            Loan_Type='Personal Loan', Loan_Amount=500000, Loan_Tenure=48, Collateral_Value=None),
    'Q. Messy input (Rs., lower-case, spaces)':        dict(Gender=' m ', Loan_Type=' HL', Loan_Amount='Rs. 30,00,000', Employment_Type='SERVICE'),
}
apps = pd.DataFrame([{**base, **v, 'Loan_ID': k} for k, v in scenarios.items()])
res = engine.decide(apps)
res[['Loan_ID', 'decision', 'decided_by', 'p_approve', 'rules_violated', 'reasons']]

## 9. Fairness audit (requirement e)

In [ ]:
frame = test_df[['Gender', 'Property_Area', 'target']].copy()
frame['Gender'] = frame['Gender'].fillna('Unknown')
frame['approved'] = (p_test >= .5).astype(int)
age_band = pd.cut(test_df['Age'], [0, 25, 35, 45, 55, 100], labels=['<=25', '26-35', '36-45', '46-55', '>55'])
frame['Age_band'] = age_band
for g in ['Gender', 'Property_Area', 'Age_band']:
    display(T.fairness_audit(frame.dropna(subset=[g]), g))

**Reading the audit:**
- **DIR** is the group's approval rate divided by the largest group's rate. The four-fifths rule flags values below 0.8.
- **TPR gap** is equal opportunity: among applicants who historically were approved, how much less often this group gets approved.
- The model never sees gender. Any remaining gap comes through legitimate factors such as the ~15% income gap in the data, not through direct bias. Section 8, scenario B, shows that changing only gender leaves the score unchanged.
- Groups with fewer than ~100 test rows (Transgender, Unknown) have wide confidence intervals and should be reported with them, not as point estimates.
- The low rates for the youngest and oldest age bands come from R1, R2 and R7, which are legitimate age-linked policy.

## 10. Train on all history and save

In [ ]:
final_engine, _ = T.main()      # prints the evaluation, writes artifacts/loan_engine.joblib and audit CSVs

In [ ]:
# Using the saved engine on a new application
import joblib
eng = joblib.load('artifacts/loan_engine.joblib')
new_app = pd.DataFrame([{**base, 'Loan_ID': 'LN-NEW-001', 'CIBIL_Score': 742, 'Monthly_Income': 85000,
                         'Existing_EMI': 9000, 'Loan_Amount': 4200000, 'Collateral_Value': 5600000}])
eng.decide(new_app).T

## Summary

| | Video approach | This engine |
|---|---|---|
| Accuracy | 0.840 (Random Forest, one split); 0.847 at best in 5-fold CV | **0.897 in 5-fold CV; 0.910 on the hold-out test set** |
| ROC-AUC | 0.909 | **0.946 in CV; 0.955 on hold-out** |
| Hybrid engine | — | **84% of files auto-decided, 94% of those match the historical decision**; 16% go to an underwriter with reasons |
| Policy guaranteed | No: the model can approve a CIBIL 600 applicant | **Yes**: R1–R7 always enforced |
| Uncertain or missing data | Silently filled with the mode or mean | **Routed to manual review** |
| Reasons for a decision | None | **Rule text, or top SHAP factors in plain language** |
| Gender used | Yes (as a feature) | **No**; audited instead |
| Leakage | `fit_transform` on the test set | **None**: everything is fitted on the training fold |
| Tested | — | **51 scenario tests** (`pytest test_scenarios.py`) |